## LABORATORIO N.° 07

## Semana 7: Regresión Lineal con datos en SQLite (estadística inferencial)

**Curso:** Fundamentos de Gestión de Datos
**Docente:** Pilar Rocío Sayán Mejía
**Modalidad:** Individual
**Nombre del estudiante:** __________________________
**Sección:** __________________________

---

# Caso de negocio

Una cadena de **abarrotes** registra sus operaciones de venta e inventario en una base de datos **SQLite** (`abarrotes_ventas_inventario.db`, tabla `ventas_original`). El área comercial desea **estimar el monto de venta en soles** a partir de variables operativas de inventario y descuento, usando **regresión lineal** e interpretando los resultados con pruebas estadísticas (no como un modelo de *machine learning*: se trabaja con **todo el conjunto de datos**, sin división en entrenamiento y prueba).

---

# Fundamento teórico

| N.° | Concepto / Principio | Definición aplicada al caso |
| --- | --- | --- |
| 1 | Variable objetivo | Variable que se desea explicar; aquí, el **monto de venta** (`monto_venta_soles`). |
| 2 | Variable predictora | Variable utilizada para explicar la variable objetivo (stock, descuento, merma, tiempo). |
| 3 | Coeficiente de regresión (pendiente) | Cambio esperado en el monto por cada unidad adicional del predictor, manteniendo las demás constantes. |
| 4 | Intercepto | Valor esperado del monto cuando todos los predictores valen cero. |
| 5 | R cuadrado (R²) | Proporción de la variabilidad del monto explicada por el modelo. |
| 6 | R cuadrado ajustado | Versión de R² que penaliza variables que no aportan capacidad explicativa. |
| 7 | p-valor | Evidencia estadística para decidir si una variable contribuye significativamente. |
| 8 | ANOVA / Prueba F | Evalúa si el modelo completo explica el monto mejor que un modelo sin predictores. |
| 9 | Residuos | Diferencia entre el monto observado y el estimado por el modelo. |
| 10 | Homocedasticidad | Supuesto de varianza constante de los residuos en todos los niveles de predicción. |

---

# Supuestos e hipótesis del modelo de regresión

Antes de interpretar la regresión lineal se formulan las hipótesis y se revisan los supuestos estadísticos mínimos. **Cada supuesto se evalúa con su hipótesis nula (H0) y alterna (H1).**

**Hipótesis sobre los coeficientes**

| Caso | H0 | H1 | Decisión |
|---|---|---|---|
| Modelo simple | La pendiente del predictor es igual a cero; no hay relación lineal significativa con el monto. | La pendiente es distinta de cero; sí hay relación lineal significativa. | Rechazar H0 si el p-valor del predictor es menor a 0.05. |
| Modelo múltiple | Todos los coeficientes de los predictores son iguales a cero. | Al menos un predictor explica significativamente el monto. | Rechazar H0 si la prueba F del modelo tiene p-valor menor a 0.05. |

**Supuestos sobre los residuos (cada uno con su prueba)**

| Supuesto | Prueba | H0 | H1 | Decisión |
|---|---|---|---|---|
| Normalidad | **Anderson-Darling** | Los residuos siguen una distribución normal. | Los residuos no siguen una distribución normal. | Se rechaza H0 si el estadístico supera el valor crítico al 5%. |
| Independencia | **Durbin-Watson** | No existe autocorrelación en los residuos (DW ≈ 2). | Existe autocorrelación en los residuos. | Independencia razonable si DW está entre 1.5 y 2.5. |
| Homocedasticidad | **Bartlett** | La varianza de los residuos es igual entre grupos. | Al menos un grupo tiene varianza distinta (heterocedasticidad). | Se rechaza H0 si el p-valor de Bartlett es menor a 0.05. |

> **Nota metodológica importante.** En esta base, `monto_venta_soles = cantidad_unidades × precio_venta_unitario` de forma **exacta**. Por eso esas columnas (y `costo_unitario`, `margen_venta_soles`) **se excluyen** del modelo: incluirlas daría un R² = 1 trivial (una identidad, no una relación estadística). El modelo se construye con variables **operativas** (stock, descuento, merma y tiempo), que sí plantean una pregunta estadística real.

## Actividad 1: Carga del dataset desde la base SQLite

In [ ]:
# Paso 1: cargar librerías y leer la tabla desde la base SQLite
import sqlite3
import urllib.request
from pathlib import Path
import sys
import subprocess

import pandas as pd
import numpy as np

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
except Exception:
    HAS_PLOTS = False

try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from statsmodels.stats.stattools import durbin_watson
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from statsmodels.stats.stattools import durbin_watson

try:
    from scipy import stats
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])
    from scipy import stats

try:
    from IPython.display import display
except Exception:
    display = print

if HAS_PLOTS:
    sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

# Base SQLite del caso (se descarga si no está disponible en el entorno)
DB_URL = (
    "https://raw.githubusercontent.com/Rociosayan/"
    "PMD1_Fundamentos_Gestion_Datos/main/casos/"
    "16_abarrotes_ventas_inventario/abarrotes_ventas_inventario.db"
)
DB_PATH = Path("abarrotes_ventas_inventario.db")

if not DB_PATH.exists():
    try:
        urllib.request.urlretrieve(DB_URL, DB_PATH)
        print("Base descargada desde el repositorio del curso.")
    except Exception as error_db:
        print("No se pudo descargar la base automáticamente. Súbela manualmente al entorno.")
        print("Detalle técnico:", error_db)

conn = sqlite3.connect(DB_PATH)

tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print("Tablas disponibles en la base SQLite:")
display(tablas)

df_original = pd.read_sql_query("SELECT * FROM ventas_original", conn)
print("Dimensiones originales:", df_original.shape)
display(df_original.head())

Pregunta 1: ¿De dónde se obtienen los datos y qué representa cada fila de la tabla `ventas_original`?

Respuesta:

Pregunta 2: ¿Por qué este caso es adecuado para aprender regresión lineal con datos provenientes de SQLite?

Respuesta:

## Actividad 2: Limpieza y exploración inicial

In [ ]:
# Paso 2: limpieza de columnas y selección de variables del modelo
df = df_original.copy()


def limpiar_numero(serie):
    """Convierte textos como '8 aprox' o '15.06' a valor numérico."""
    extraido = serie.astype(str).str.extract(r"(-?\d+(?:\.\d+)?)")[0]
    return pd.to_numeric(extraido, errors="coerce")


# Variable objetivo y predictores NUMÉRICOS (operativos, no componentes del monto)
objetivo = "monto_venta_soles"
predictores = ["descuento_pct", "stock_inicial", "stock_final", "merma_unidades"]

for col in [objetivo] + predictores:
    df[col] = limpiar_numero(df[col])

# Variables temporales derivadas de la fecha de operación
df["fecha_operacion_dt"] = pd.to_datetime(df["fecha_operacion"], errors="coerce", dayfirst=True)
df["mes_operacion"] = df["fecha_operacion_dt"].dt.month
df["fin_semana"] = df["fecha_operacion_dt"].dt.dayofweek.isin([5, 6]).astype(int)
predictores_tiempo = ["mes_operacion", "fin_semana"]

columnas_modelo = [objetivo] + predictores + predictores_tiempo
df = df.dropna(subset=columnas_modelo).copy()

print("Filas disponibles para el modelo:", df.shape[0])

print("\nResumen estadístico de las variables del modelo:")
display(df[columnas_modelo].describe())

print("Valores faltantes por columna del modelo:")
display(df[columnas_modelo].isna().sum().to_frame("nulos"))

Pregunta 3: Escribe el significado de las columnas del modelo: `monto_venta_soles`, `descuento_pct`, `stock_inicial`, `stock_final`, `merma_unidades`, `mes_operacion` y `fin_semana`.

Respuesta:

Pregunta 4: ¿La base tenía valores faltantes o textos no numéricos? ¿Qué parte del código permite afirmarlo?

Respuesta:

## Actividad 3: Supuesto de linealidad

In [ ]:
# Paso 3: revisar el supuesto de linealidad con gráficos de dispersión
predictores_analisis = ["stock_inicial", "stock_final", "descuento_pct"]

if HAS_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, col in zip(axes, predictores_analisis):
        sns.regplot(
            data=df, x=col, y=objetivo, ax=ax,
            scatter_kws={"alpha": 0.5}, line_kws={"color": "red"}
        )
        ax.set_title(f"{objetivo} vs {col}")
    plt.tight_layout()
    plt.show()
else:
    print("En Google Colab se mostrarán los gráficos de dispersión con línea de tendencia.")

correlaciones = (
    df[predictores + predictores_tiempo + [objetivo]]
    .corr(numeric_only=True)[objetivo]
    .sort_values(ascending=False)
)
print("Correlación de cada variable con el monto de venta:")
display(correlaciones.to_frame("correlación_con_monto"))

Pregunta 5: Según los gráficos de dispersión, ¿qué variable parece tener la relación lineal más clara con el monto?

Respuesta:

Pregunta 6: ¿Qué indica la correlación de cada predictor con el monto de venta?

Respuesta:

## Actividad 4: Regresión lineal simple

Se ajusta un modelo de **regresión lineal simple** por cada predictor (una sola variable a la vez) para medir su aporte individual.

In [ ]:
# Paso 4: construir modelos de regresión lineal simple (uno por predictor)
modelos_simples = {}

for predictor in predictores_analisis:
    modelo = smf.ols(f"{objetivo} ~ {predictor}", data=df).fit()
    modelos_simples[predictor] = modelo
    print("\n" + "=" * 70)
    print(f"Modelo simple: {objetivo} ~ {predictor}")
    print(f"R cuadrado: {modelo.rsquared:.4f}")
    print(f"R cuadrado ajustado: {modelo.rsquared_adj:.4f}")
    print(f"Pendiente de {predictor}: {modelo.params[predictor]:.4f}")
    print(f"p-valor de {predictor}: {modelo.pvalues[predictor]:.6f}")

resumen_simples = pd.DataFrame({
    "modelo": list(modelos_simples.keys()),
    "r_cuadrado": [m.rsquared for m in modelos_simples.values()],
    "r_cuadrado_ajustado": [m.rsquared_adj for m in modelos_simples.values()],
    "pendiente": [m.params[pred] for pred, m in modelos_simples.items()],
    "p_valor_pendiente": [m.pvalues[pred] for pred, m in modelos_simples.items()],
})
display(resumen_simples.sort_values("r_cuadrado", ascending=False))

### Hipótesis para los modelos simples

- **H0:** la pendiente del predictor es igual a cero; ese factor no explica el monto de venta.
- **H1:** la pendiente del predictor es distinta de cero; ese factor sí explica el monto.
- **Criterio:** si `p_valor_pendiente < 0.05`, se rechaza H0.

Se espera que `stock_inicial` sea el predictor simple con mayor R², mientras que `descuento_pct` aporte poco por sí solo.

Pregunta 7: ¿Cuál de los modelos simples tiene mayor R cuadrado? Interpreta qué significa.

Respuesta:

Pregunta 8: Interpreta la pendiente del modelo con `stock_inicial`. ¿Qué significa en contexto de negocio?

Respuesta:

Pregunta 9: ¿Qué variable tiene menor aporte como modelo simple? Justifica con R cuadrado y p-valor.

Respuesta:

## Actividad 5: Regresión lineal múltiple y selección del mejor modelo

Se ajusta primero el **modelo múltiple completo** con todas las variables operativas. Luego, mediante **eliminación hacia atrás**, se quitan las variables que **no aportan** (p-valor ≥ 0.05) hasta quedarnos con el **mejor modelo**.

In [ ]:
# Paso 5a: modelo de regresión múltiple completo (todas las variables operativas)
predictores_multiple = [
    "descuento_pct", "stock_inicial", "stock_final",
    "merma_unidades", "mes_operacion", "fin_semana",
]

modelo_completo = smf.ols(
    f"{objetivo} ~ " + " + ".join(predictores_multiple), data=df
).fit()

print(modelo_completo.summary())

# Identificar las variables que NO aportan (p-valor >= 0.05)
pvalores = modelo_completo.pvalues.drop("Intercept").sort_values(ascending=False)
print("\np-valores de cada variable (de mayor a menor):")
display(pvalores.to_frame("p_valor"))

no_aportan = list(pvalores[pvalores >= 0.05].index)
si_aportan = list(pvalores[pvalores < 0.05].index)
print("Variables que NO aportan (se eliminan):", no_aportan)
print("Variables significativas (se conservan):", si_aportan)

In [ ]:
# Paso 5b: eliminar las variables que no aportan -> MEJOR MODELO (múltiple final)
modelo_best = smf.ols(f"{objetivo} ~ stock_inicial + stock_final", data=df).fit()

print(modelo_best.summary())

tabla_coeficientes = pd.DataFrame({
    "coeficiente": modelo_best.params,
    "p_valor": modelo_best.pvalues,
    "intervalo_95_inf": modelo_best.conf_int()[0],
    "intervalo_95_sup": modelo_best.conf_int()[1],
})
print("Coeficientes del mejor modelo:")
display(tabla_coeficientes)

print(f"R cuadrado: {modelo_best.rsquared:.4f}")
print(f"R cuadrado ajustado: {modelo_best.rsquared_adj:.4f}")
print(f"Prueba F del modelo - p-valor: {modelo_best.f_pvalue:.8f}")

### Hipótesis del modelo múltiple y eliminación de variables

En el modelo completo solo `stock_inicial` y `stock_final` resultan significativas; `descuento_pct`, `merma_unidades`, `mes_operacion` y `fin_semana` tienen p-valor ≥ 0.05, por lo que **no aportan** y se eliminan. El **mejor modelo** queda:

`monto_venta_soles ~ stock_inicial + stock_final`

- **H0:** `stock_inicial` y `stock_final` no explican el monto de venta.
- **H1:** al menos una de las dos variables explica el monto.

Como la prueba F del mejor modelo tiene p-valor ≈ 0.0000, se **rechaza H0**. Los signos opuestos (stock_inicial positivo, stock_final negativo) tienen sentido: `stock_inicial − stock_final` aproxima las **unidades vendidas**, que impulsan el monto.

Pregunta 10: Interpreta los coeficientes de `stock_inicial` y `stock_final`. ¿Por qué tienen signos opuestos?

Respuesta:

Pregunta 11: ¿Qué variables se eliminaron del modelo múltiple y por qué? Usa los p-valores como criterio.

Respuesta:

Pregunta 12: ¿Para qué sirve el R cuadrado ajustado al comparar el modelo completo con el mejor modelo?

Respuesta:

## Actividad 6: ANOVA y prueba F del mejor modelo

In [ ]:
# Paso 6: ANOVA del mejor modelo y aporte de cada variable
anova_modelo = sm.stats.anova_lm(modelo_best, typ=2)
print("ANOVA del mejor modelo de regresión:")
display(anova_modelo)

print("Interpretación rápida:")
print("- Si PR(>F) es menor que 0.05, la variable aporta evidencia estadística dentro del modelo.")
print("- Permite comparar cuánto explica cada predictor del monto de venta.")

Pregunta 13: ¿Qué evalúa la prueba F o ANOVA en el contexto de este modelo?

Respuesta:

Pregunta 14: Según la tabla ANOVA, ¿qué variable aporta más evidencia para explicar el monto de venta?

Respuesta:

## Actividad 7: Diagnóstico de residuos y prueba de supuestos

Sobre el **mejor modelo** se revisan los tres supuestos clásicos de los residuos. Cada uno se presenta con su **hipótesis nula (H0)** y **alterna (H1)**, su estadístico y la decisión:

- **Normalidad → Anderson-Darling**
- **Independencia → Durbin-Watson**
- **Homocedasticidad → Bartlett**

In [ ]:
# Paso 7: diagnóstico gráfico de los residuos del mejor modelo
df_diagnostico = df.copy()
df_diagnostico["monto_predicho"] = modelo_best.fittedvalues
df_diagnostico["residuo"] = modelo_best.resid

print("Primeras predicciones y residuos:")
display(df_diagnostico[[objetivo, "monto_predicho", "residuo"]].head())

if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df_diagnostico["residuo"], kde=True, ax=axes[0], color="steelblue")
    axes[0].set_title("Distribución de residuos")
    sm.qqplot(df_diagnostico["residuo"], line="45", fit=True, ax=axes[1])
    axes[1].set_title("QQ plot de residuos")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4))
    sns.scatterplot(data=df_diagnostico, x="monto_predicho", y="residuo")
    plt.axhline(0, color="red", linestyle="--")
    plt.title("Residuos vs valores predichos")
    plt.xlabel("Monto predicho (S/)")
    plt.ylabel("Residuo")
    plt.tight_layout()
    plt.show()
else:
    print("En Google Colab se mostrarán histograma, QQ plot y residuos vs predichos.")

In [ ]:
# Paso 7b: pruebas formales de los supuestos, cada una con H0 y H1
residuos = modelo_best.resid

# ---------- SUPUESTO 1: NORMALIDAD (Anderson-Darling) ----------
ad = stats.anderson(residuos, dist="norm")
idx_5 = list(ad.significance_level).index(5.0)
critico_5 = ad.critical_values[idx_5]
normal_ok = ad.statistic <= critico_5

print("SUPUESTO 1 - NORMALIDAD DE LOS RESIDUOS  (prueba de Anderson-Darling)")
print("  H0: los residuos siguen una distribución normal.")
print("  H1: los residuos NO siguen una distribución normal.")
print(f"  Estadístico A-D = {ad.statistic:.4f} | valor crítico al 5% = {critico_5:.4f}")
print("  Decisión:", "No se rechaza H0 -> normalidad" if normal_ok
      else "Se rechaza H0 -> los residuos no son normales (limitación)")

# ---------- SUPUESTO 2: INDEPENDENCIA (Durbin-Watson) ----------
dw = durbin_watson(residuos)
indep_ok = 1.5 <= dw <= 2.5

print("\nSUPUESTO 2 - INDEPENDENCIA DE LOS RESIDUOS  (prueba de Durbin-Watson)")
print("  H0: no existe autocorrelación en los residuos (DW cercano a 2).")
print("  H1: existe autocorrelación en los residuos.")
print(f"  Estadístico Durbin-Watson = {dw:.4f}")
print("  Decisión:", "No se rechaza H0 -> independencia razonable" if indep_ok
      else "Se rechaza H0 -> posible autocorrelación")

# ---------- SUPUESTO 3: HOMOCEDASTICIDAD (Bartlett) ----------
# Se agrupan los residuos en cuartiles del valor predicho y se comparan sus varianzas.
df_diagnostico["grupo_predicho"] = pd.qcut(
    df_diagnostico["monto_predicho"], q=4, labels=["G1", "G2", "G3", "G4"]
)
grupos_residuo = [
    g["residuo"].values
    for _, g in df_diagnostico.groupby("grupo_predicho", observed=True)
]
bart_stat, bart_p = stats.bartlett(*grupos_residuo)
homo_ok = bart_p > 0.05

print("\nSUPUESTO 3 - HOMOCEDASTICIDAD  (prueba de Bartlett, residuos por cuartiles del predicho)")
print("  H0: la varianza de los residuos es igual en todos los grupos (homocedasticidad).")
print("  H1: al menos un grupo tiene varianza distinta (heterocedasticidad).")
print(f"  Estadístico de Bartlett = {bart_stat:.4f} | p-valor = {bart_p:.6f}")
print("  Decisión:", "No se rechaza H0 -> varianza constante" if homo_ok
      else "Se rechaza H0 -> heterocedasticidad (limitación)")

print("\nResumen de supuestos:")
display(pd.DataFrame({
    "supuesto": ["Normalidad", "Independencia", "Homocedasticidad"],
    "prueba": ["Anderson-Darling", "Durbin-Watson", "Bartlett"],
    "estadístico": [round(ad.statistic, 4), round(dw, 4), round(bart_stat, 4)],
    "referencia": [f"crítico 5% = {critico_5:.4f}", "rango 1.5 - 2.5", f"p-valor = {bart_p:.4f}"],
    "se_cumple": [normal_ok, indep_ok, homo_ok],
}))

### Decisión de supuestos del mejor modelo

| Supuesto | Prueba | H0 | Resultado esperado | Decisión |
|---|---|---|---|---|
| Normalidad | Anderson-Darling | Residuos normales | Estadístico mayor al valor crítico 5% | Se rechaza H0; la normalidad no se cumple (limitación). |
| Independencia | Durbin-Watson | Sin autocorrelación | DW cercano a 2 | No se rechaza H0; independencia razonable. |
| Homocedasticidad | Bartlett | Varianza constante | p-valor muy bajo | Se rechaza H0; hay heterocedasticidad (limitación). |

El modelo conserva valor descriptivo (predictores significativos y R² ≈ 0.50), pero los incumplimientos de **normalidad** y **homocedasticidad** deben mencionarse como **limitaciones** en las conclusiones.

Pregunta 15: Según Anderson-Darling, ¿los residuos son normales? Indica H0, H1 y tu decisión.

Respuesta:

Pregunta 16: Según Bartlett, ¿se cumple la homocedasticidad? Indica H0, H1 y tu decisión.

Respuesta:

Pregunta 17: Según Durbin-Watson, ¿hay independencia en los residuos? Interpreta el valor obtenido.

Respuesta:

## Actividad 8: Predicción e interpretación ejecutiva

In [ ]:
# Paso 8: predicción del monto de venta para un escenario de inventario
nuevo_escenario = pd.DataFrame({
    "stock_inicial": [80],
    "stock_final": [20],
})

prediccion = modelo_best.get_prediction(nuevo_escenario).summary_frame(alpha=0.05)

print("Escenario propuesto (stock inicial 80, stock final 20):")
display(nuevo_escenario)

print("Predicción del monto de venta con intervalo de confianza y de predicción:")
display(prediccion)

Pregunta 18: Interpreta la predicción del escenario propuesto. ¿Qué monto de venta esperaría la tienda?

Respuesta:

Pregunta 19: En cinco líneas, recomienda qué debería vigilar el negocio (stock, rotación, descuentos) según el modelo y por qué.

Respuesta:

## CONCLUSIONES

| 1. |  |
|---|---|
| 2. |  |
| 3. |  |

## MATERIAL COMPLEMENTARIO

- Base del caso: `abarrotes_ventas_inventario.db` (tabla `ventas_original`), curso Fundamentos de Gestión de Datos.
- An Introduction to Statistical Learning.
- Statsmodels documentation: https://www.statsmodels.org/
- SciPy stats (Anderson-Darling, Bartlett): https://docs.scipy.org/doc/scipy/reference/stats.html
- SQLite documentation: https://www.sqlite.org/docs.html